<a href="https://colab.research.google.com/github/Scrap263/work_vk_to_telegram/blob/main/flight_delay_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✈️ Flight Delay — Blending (XGBoost × 5, без Optuna)

**Идея:** не тратить время на подбор гиперпараметров.5 моделей XGBoost с разными random_state, среднее предсказаний.Все 27 признаков (никакого дропа).

## 1. Импорт

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

pd.set_option('display.max_columns', 40)

## 2. Загрузка

In [2]:
from google.colab import files
uploaded = files.upload()

train = pd.read_csv('flight_delays_train.csv')
test = pd.read_csv('flight_delays_test.csv')
sample = pd.read_csv('sample_submission.csv')

Saving flight_delays_test.csv to flight_delays_test.csv
Saving flight_delays_train.csv to flight_delays_train.csv
Saving sample_submission.csv to sample_submission.csv


## 3. Препроцессинг + все фичи

In [3]:
X = train.copy()
X_test = test.copy()

# c- → int
for col in ['Month', 'DayofMonth', 'DayOfWeek']:
    for df in [X, X_test]:
        df[col+'_num'] = df[col].str.replace('c-','').astype(int)
        df.drop(columns=[col], inplace=True)

# DepTime
for df in [X, X_test]:
    df['DepHour'] = df['DepTime']//100
    df['DepMin'] = df['DepTime']%100
    df.drop(columns=['DepTime'], inplace=True)

# Carrier
X['CarrierStr'] = X['UniqueCarrier']
X_test['CarrierStr'] = X_test['UniqueCarrier']
carrier_map = {c:i for i,c in enumerate(X['UniqueCarrier'].unique())}
max_code = max(carrier_map.values())
X['CarrierCode'] = X['UniqueCarrier'].map(carrier_map)
X_test['CarrierCode'] = X_test['UniqueCarrier'].map(carrier_map).fillna(max_code+1).astype(int)
for df in [X, X_test]:
    df.drop(columns=['UniqueCarrier'], inplace=True)

# Origin/Dest
for col in ['Origin', 'Dest']:
    X[col+'_str'] = train[col]
    X_test[col+'_str'] = test[col]
    freq = X[col+'_str'].value_counts()
    X[col+'_freq'] = X[col+'_str'].map(freq)
    X_test[col+'_freq'] = X_test[col+'_str'].map(freq).fillna(freq.median())
    for df in [X, X_test]:
        df.drop(columns=[col], inplace=True)

# Таргет
y = (X['dep_delayed_15min']=='Y').astype(int)
X.drop(columns=['dep_delayed_15min'], inplace=True)

# Cyclical encoding
for df in [X, X_test]:
    df['Month_sin'] = np.sin(2*np.pi*df['Month_num']/12)
    df['Month_cos'] = np.cos(2*np.pi*df['Month_num']/12)
    df['DoW_sin'] = np.sin(2*np.pi*df['DayOfWeek_num']/7)
    df['DoW_cos'] = np.cos(2*np.pi*df['DayOfWeek_num']/7)
    df['Day_sin'] = np.sin(2*np.pi*df['DayofMonth_num']/31)
    df['Day_cos'] = np.cos(2*np.pi*df['DayofMonth_num']/31)
    df['Hour_sin'] = np.sin(2*np.pi*df['DepHour']/24)
    df['Hour_cos'] = np.cos(2*np.pi*df['DepHour']/24)

# Target Encoding
GLOBAL_MEAN = y.mean()
for col in ['CarrierStr', 'Origin_str', 'Dest_str']:
    te_train = np.full(len(X), GLOBAL_MEAN)
    te_test = np.zeros(len(X_test))
    skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for tr_idx, vl_idx in skf_te.split(X, y):
        means = y.iloc[tr_idx].groupby(X[col].iloc[tr_idx]).mean()
        te_train[vl_idx] = X[col].iloc[vl_idx].map(means).fillna(GLOBAL_MEAN)
    global_means = y.groupby(X[col]).mean()
    te_test = X_test[col].map(global_means).fillna(GLOBAL_MEAN)
    col_name = col.replace('_str','_te')
    X[col_name] = te_train
    X_test[col_name] = te_test
for df in [X, X_test]:
    df.drop(columns=['CarrierStr','Origin_str','Dest_str'], inplace=True)

# Бинарные флаги + взаимодействия
for df in [X, X_test]:
    df['is_rush_hour'] = df['DepHour'].isin([7,8,9,17,18,19]).astype(int)
    df['is_weekend'] = df['DayOfWeek_num'].isin([6,7]).astype(int)
    df['is_december'] = (df['Month_num']==12).astype(int)
    df['Carrier_rush'] = df['CarrierCode'] * df['is_rush_hour']
    df['Month_Day'] = df['Month_num'] * df['DayOfWeek_num']
    df['Dist_per_hour'] = df['Distance'] / (df['DepHour'] + 1)

# Hub ratio
train_orig = pd.read_csv('flight_delays_train.csv')
test_orig = pd.read_csv('flight_delays_test.csv')
origin_cnt = train_orig['Origin'].value_counts()
dest_cnt = train_orig['Dest'].value_counts()
all_ap = set(origin_cnt.index) | set(dest_cnt.index)
hub_ratio = {ap: (origin_cnt.get(ap,0)+1)/(dest_cnt.get(ap,0)+1) for ap in all_ap}
X['Origin_hub'] = train_orig['Origin'].map(hub_ratio)
X['Dest_hub'] = train_orig['Dest'].map(hub_ratio)
X_test['Origin_hub'] = test_orig['Origin'].map(hub_ratio).fillna(1.0)
X_test['Dest_hub'] = test_orig['Dest'].map(hub_ratio).fillna(1.0)

print(f'Фич: {X.shape[1]}')

Фич: 27


## 4. Blending: 5 XGBoost (дефолтные параметры)

In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Дефолтные параметры XGBoost (хороший баланс скорость/качество)
BASE_PARAMS = {
    'n_estimators': 200,
    'max_depth': 20,
    'learning_rate': 0.03,
    'subsample': 0.65,
    'colsample_bytree': 0.2,
    'reg_alpha': 3,
    'scale_pos_weight': (y_train==0).sum() / (y_train==1).sum(),
    'eval_metric': 'auc',
    'n_jobs': -1,
    'verbosity': 0
}

N_MODELS = 15
seeds = [42, 123, 456, 789, 1024, 342, 84, 688, 667, 291, 831, 518, 999, 109, 190]
val_preds = np.zeros((len(X_val), N_MODELS))
test_preds = np.zeros((len(X_test), N_MODELS))

print(f'Обучаю {N_MODELS} XGBoost...\n')
for i, seed in enumerate(seeds):
    m = xgb.XGBClassifier(**BASE_PARAMS, random_state=seed)
    m.fit(X_train, y_train)
    val_preds[:,i] = m.predict_proba(X_val)[:,1]
    test_preds[:,i] = m.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_val, val_preds[:,i])
    print(f'  Model {i+1} (seed={seed:4d}): AUC = {auc:.4f}')

val_blend = val_preds.mean(axis=1)
auc_blend = roc_auc_score(y_val, val_blend)
best_solo = max(roc_auc_score(y_val, val_preds[:,i]) for i in range(N_MODELS))

print(f'\nЛучшая одиночная:  {best_solo:.4f}')
print(f'Blending (5):       {auc_blend:.4f}')
print(f'Дельта:             +{(auc_blend-best_solo)*100:.2f}%')

Обучаю 15 XGBoost...

  Model 1 (seed=  42): AUC = 0.7544
  Model 2 (seed= 123): AUC = 0.7531
  Model 3 (seed= 456): AUC = 0.7545
  Model 4 (seed= 789): AUC = 0.7557
  Model 5 (seed=1024): AUC = 0.7564
  Model 6 (seed= 342): AUC = 0.7570
  Model 7 (seed=  84): AUC = 0.7557
  Model 8 (seed= 688): AUC = 0.7582
  Model 9 (seed= 667): AUC = 0.7559
  Model 10 (seed= 291): AUC = 0.7549
  Model 11 (seed= 831): AUC = 0.7524
  Model 12 (seed= 518): AUC = 0.7522
  Model 13 (seed= 999): AUC = 0.7556
  Model 14 (seed= 109): AUC = 0.7560
  Model 15 (seed= 190): AUC = 0.7561

Лучшая одиночная:  0.7582
Blending (5):       0.7614
Дельта:             +0.32%


## 5. Сабмишен

In [5]:
test_blend = test_preds.mean(axis=1)
submission = pd.DataFrame({'id': sample['id'], 'dep_delayed_15min': test_blend})
submission.to_csv('submission.csv', index=False)
print(submission.head())
files.download('submission.csv')

   id  dep_delayed_15min
0   0           0.142230
1   1           0.173728
2   2           0.150879
3   3           0.580107
4   4           0.567138


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>